In [1]:
import re
import time
import spacy
import sqlite3
import inflect
import pickle
import pandas as pd
from pandas import DataFrame
from taxonerd import TaxoNERD
from IPython.display import clear_output
from rich.console import Console
from spacy.util import filter_spans
from spacy.tokens import Token, Doc, Span
from typing import List, Dict, Tuple, Set, Optional, Union, Literal

In [2]:
inflector = inflect.engine()

In [3]:
conn = sqlite3.connect("COL.db")

conn.execute('CREATE INDEX IF NOT EXISTS IndexID ON NameUsage ("col:ID")')

conn.execute('CREATE INDEX IF NOT EXISTS IndexRank ON NameUsage ("col:rank")')

conn.execute('CREATE INDEX IF NOT EXISTS IndexGenusLowerCase ON NameUsage (LOWER("col:genus"))')

conn.execute('CREATE INDEX IF NOT EXISTS IndexScientificNameLowerCase ON NameUsage (LOWER("col:scientificName"))')

conn.execute('CREATE INDEX IF NOT EXISTS IndexGenericNameLowerCase ON NameUsage (LOWER("col:genericName"))')

conn.execute('CREATE INDEX IF NOT EXISTS IndexGenericNameLowerCaseFirstCharacter ON NameUsage (SUBSTR(LOWER("col:genericName"), 1, 1))')

conn.execute('CREATE INDEX IF NOT EXISTS IndexSpecificEpithetLowerCase ON NameUsage (LOWER("col:specificEpithet"))')

conn.execute('CREATE INDEX IF NOT EXISTS IndexNameLowerCase ON VernacularName (LOWER("col:name"))')

conn.execute('CREATE INDEX IF NOT EXISTS IndexTaxonID ON VernacularName ("col:taxonID")')

conn.execute('CREATE INDEX IF NOT EXISTS IndexLanguage ON VernacularName ("col:language")')

conn.commit()

In [4]:
console = Console()

def highlight(string, substrings):
    for substring in substrings:
        string = re.sub(f"({re.escape(substring)})", r"[bold black on bright_yellow]\1[/bold black on bright_yellow]", string, flags=re.IGNORECASE)
    console.print(string)

In [5]:
def re_is_scientific_name(name: str, flags: int = 0) -> bool:
    return re.match(r"^[A-Z][a-z]+\s[a-z]+$", name, flags)

In [6]:
def re_is_scientific_name_abbrev(name: str, flags: int = 0) -> Union[str, Tuple[str, str]]:
    if not re.match(r"^[A-Z]\.\s[a-z]+$", name, flags):
        return False
    return name.split()

In [7]:
def re_is_genus(name: str, flags: int = 0) -> bool:
    return re.match(r"^[A-Z][a-z]+$", name, flags)

In [8]:
def abbreviate_scientific_name(name: str) -> Union[bool, str]:
    if not re_is_scientific_name(name, flags=re.IGNORECASE):
        return False
    name = name.split()
    return f"{name[0][0].upper()}. {name[1].lower()}"

In [9]:
def clean_query_input(string: str) -> str:
    return re.sub(r"'", "", string)

In [10]:
def find_genus_query(name: Union[str, Tuple[str, str]]):
    return f"""
        SELECT NameUsage."col:genus"
        FROM NameUsage
        WHERE LOWER("col:genus") = LOWER('{clean_query_input(name)}')
        LIMIT 1
    """

In [11]:
def find_scientific_name_query(name: Union[str, Tuple[str, str]], only_species: bool = False):
    where_species = "" if not only_species else 'AND NameUsage."col:rank" = \'species\''
    where = None
    if isinstance(name, str):
        where = f'WHERE LOWER(NameUsage."col:scientificName") = LOWER(\'{clean_query_input(name)}\')'
    else:
        where = f'WHERE SUBSTR(LOWER(NameUsage."col:genericName"), 1, 1) = LOWER(\'{clean_query_input(name[0][0])}\') AND LOWER(NameUsage."col:specificEpithet") = LOWER(\'{clean_query_input(name[1])}\')'
    
    return f"""
        SELECT NameUsage."col:scientificName"
        FROM NameUsage
        {where}
        {where_species}
        LIMIT 1
    """

In [12]:
def find_vernacular_name_query(name: str, only_english: bool = False):
    where_english = "" if not only_english else 'AND VernacularName."col:language" = \'eng\''
    
    return f"""
        SELECT VernacularName."col:name"
        FROM VernacularName
        WHERE LOWER(VernacularName."col:name") = LOWER('{clean_query_input(name)}')
        {where_english}
        LIMIT 1
    """

In [13]:
def find_scientific_name_subs_query(name: Union[str, Tuple[str, str]], only_species: bool = False, only_english: bool = False):
    where = None
    if isinstance(name, str):
        where = f'WHERE LOWER(NameUsage."col:scientificName") = LOWER(\'{clean_query_input(name)}\')'
    else:
        where = f'WHERE SUBSTR(LOWER(NameUsage."col:genericName"), 1, 1) = LOWER(\'{clean_query_input(name[0][0])}\') AND LOWER(NameUsage."col:specificEpithet") = LOWER(\'{clean_query_input(name[1])}\')'

    where_species = "" if not only_species else 'AND NameUsage."col:rank" = \'species\''
    where_english = "" if not only_english else 'AND VernacularName."col:language" = \'eng\''
    
    return f"""
        SELECT NameUsage."col:scientificName", VernacularName."col:name", NameUsage."col:genus"
        FROM NameUsage
        JOIN VernacularName 
        ON NameUsage."col:ID" = VernacularName."col:taxonID"
        {where}
        {where_species}
        {where_english}
        LIMIT 100
    """

In [14]:
def find_vernacular_name_subs_query(name: str, only_species: bool = False, only_english: bool = False):
    where_species = "" if not only_species else 'AND NameUsage."col:rank" = \'species\''
    where_english = "" if not only_english else 'AND VernacularName."col:language" = \'eng\''
    
    return f"""
        SELECT NameUsage."col:scientificName", VernacularName."col:name", NameUsage."col:genus"
        FROM NameUsage
        JOIN VernacularName 
        ON NameUsage."col:ID" = VernacularName."col:taxonID"
        WHERE LOWER(VernacularName."col:name") = LOWER('{clean_query_input(name)}')
        {where_species}
        {where_english}
        LIMIT 100
    """

In [15]:
def find_genus_subs_query(name: str):
    return f"""
        SELECT NameUsage."col:scientificName", VernacularName."col:name", NameUsage."col:genus"
        FROM NameUsage
        JOIN VernacularName 
        ON NameUsage."col:ID" = VernacularName."col:taxonID"
        WHERE LOWER(NameUsage."col:genus") = LOWER('{clean_query_input(name)}')
        LIMIT 100
    """

In [16]:
CACHE_SEARCH_SCIENTIFIC_NAME = {}
def search_scientific_name(name: str) -> DataFrame:
    if name in CACHE_SEARCH_SCIENTIFIC_NAME:
        return CACHE_SEARCH_SCIENTIFIC_NAME[name]
    
    fname = re_is_scientific_name_abbrev(name) or name
    query = find_scientific_name_query(fname)
    output = pd.read_sql(query, conn)
    CACHE_SEARCH_SCIENTIFIC_NAME[name] = output
    
    return output

In [17]:
CACHE_SEARCH_VERNACULAR_NAME = {}
def search_vernacular_name(name: str) -> DataFrame:
    if name in CACHE_SEARCH_VERNACULAR_NAME:
        return CACHE_SEARCH_VERNACULAR_NAME[name]
    
    query = find_vernacular_name_query(name)
    output = pd.read_sql(query, conn)
    CACHE_SEARCH_VERNACULAR_NAME[name] = output
    
    return output

In [18]:
CACHE_SEARCH_GENUS = {}
def search_genus(name: str) -> DataFrame:
    if name in CACHE_SEARCH_GENUS:
        return CACHE_SEARCH_GENUS[name]
    
    query = find_genus_query(name)
    output = pd.read_sql(query, conn)
    CACHE_SEARCH_GENUS[name] = output
    
    return output

In [19]:
CACHE_FIND_SCIENTIFIC_NAME_SUBSTITUTIONS = {}

def find_scientific_name_substitutions(name: str) -> list[str]:
    if name in CACHE_FIND_SCIENTIFIC_NAME_SUBSTITUTIONS:
        return CACHE_FIND_SCIENTIFIC_NAME_SUBSTITUTIONS[name]

    fname = re_is_scientific_name_abbrev(name) or name
    query = find_scientific_name_subs_query(fname)
    output = pd.read_sql(query, conn)
    
    # Get Substitutions
    names = output["col:name"].tolist()
    genera = output["col:genus"].tolist()
    
    output = [*names, *genera]
    output = [text.lower() for text in output if text]
    output = [*set(output)]

    CACHE_FIND_SCIENTIFIC_NAME_SUBSTITUTIONS[name] = output
    return output

In [20]:
CACHE_FIND_VERNACULAR_NAME_SUBSTITUTIONS = {}

def find_vernacular_name_substitutions(name: str) -> list[str]:
    if name in CACHE_FIND_VERNACULAR_NAME_SUBSTITUTIONS:
        return CACHE_FIND_VERNACULAR_NAME_SUBSTITUTIONS[name]
    
    query = find_vernacular_name_subs_query(name)
    output = pd.read_sql(query, conn)

    # Get Substitutions
    scientific_names = output["col:scientificName"].tolist()
    genera = output["col:genus"].tolist()
    
    output = [*scientific_names, *genera]
    output = [text.lower() for text in output if text]
    output = [*set(output)]

    CACHE_FIND_VERNACULAR_NAME_SUBSTITUTIONS[name] = output
    return output

In [21]:
CACHE_FIND_GENUS_SUBSTITUTIONS = {}

def find_genus_substitutions(name: str) -> list[str]:
    if name in CACHE_FIND_GENUS_SUBSTITUTIONS:
        return CACHE_FIND_GENUS_SUBSTITUTIONS[name]
    
    query = find_genus_subs_query(name)
    output = pd.read_sql(query, conn)

    # Get Substitutions
    scientific_names = output["col:scientificName"].tolist()
    vernacular_names = output["col:name"].tolist()
    
    output = [*scientific_names, *vernacular_names]
    output = [text.lower() for text in output if text]
    output = [*set(output)]

    CACHE_FIND_GENUS_SUBSTITUTIONS[name] = output
    return output

In [22]:
def found_scientific_name(name: str) -> bool:
    if not re_is_scientific_name(name) and not re_is_scientific_name_abbrev(name):
        return False
    return not search_scientific_name(name).empty

In [23]:
def found_vernacular_name(name: str) -> bool:
    return not search_vernacular_name(name).empty

In [24]:
def found_genus(name: str) -> bool:
    return not search_genus(name).empty

In [25]:
def expand_unit(doc: Doc, il_unit: int, ir_unit: int, il_boundary: int, ir_boundary: int, speech: List[str] = [], literals: List[str] =[], include: bool = True, direction: str = 'BOTH', verbose: bool = False):
        UNIT = doc[il_unit:ir_unit+1]
        
        if il_unit > ir_unit:
            print(f"Error: il_unit of {il_unit} greater than ir_unit of {ir_unit}")
            return None
        
        if direction in ['BOTH', 'LEFT'] and il_boundary > il_unit:
            print(f"Error: il_unit of {il_unit} less than il_boundary of {il_boundary}")
            return None
        
        if direction in ['BOTH', 'RIGHT'] and ir_boundary < ir_unit:
            print(f"Error: ir_unit of {ir_unit} greater than ir_boundary of {ir_boundary}")
            return None
        
        # Move Left
        if direction in ['BOTH', 'LEFT']:
            # The indices are inclusive, therefore, when 
            # the condition fails, il_unit will be equal
            # to il_boundary.
            while il_unit > il_boundary:
                # We assume that the current token is allowed,
                # and look to the token to the left.
                l_token = doc[il_unit-1]

                # If the token is invalid, we stop expanding.
                in_set = l_token.pos_ in speech or l_token.lower_ in literals

                # Case 1: include=False, in_set=True
                # If we're not meant to include the defined tokens, and the
                # current token is in that set, we stop expanding.
                # Case 2: include=True, in_set=False
                # If we're meant to include the defined tokens, and the current
                # token is not in that set, we stop expanding.
                # Case 3: include=in_set
                # If we're meant to include the defined tokens, and the current
                # token is in that set, we continue expanding. If we're not meant
                # to include the defined tokens, and the current token is not
                # in that set, we continue expanding.
                if include ^ in_set:
                    break
                
                # Else, the left token is valid, and
                # we continue to expand.
                il_unit -= 1

        # Move Right
        if direction in ['BOTH', 'RIGHT']:
            # Likewise, when the condition fails,
            # ir_unit will be equal to the ir_boundary.
            # The ir_boundary is also inclusive.
            while ir_unit < ir_boundary:
                # Assuming that the current token is valid,
                # we look to the right to see if we can
                # expand.
                r_token = doc[ir_unit+1]

                # If the token is invalid, we stop expanding.
                in_set = r_token.pos_ in speech or r_token.lower_ in literals
                if include ^ in_set:
                    break

                # Else, the token is valid and
                # we continue.
                ir_unit += 1

        assert il_unit >= il_boundary and ir_unit <= ir_boundary
        expanded_unit = doc[il_unit:ir_unit+1]

        if verbose and VERBOSE_LEVEL >= 1:
            print(f"Expanded Unit of '{UNIT}': {expanded_unit}")
        
        return expanded_unit

In [26]:
def contract_unit(doc: Doc, il_unit: int, ir_unit: int, speech: List[str] = [], literals: List[str] =[], include: bool =True, direction: str = 'BOTH', verbose: bool =False):
        UNIT = doc[il_unit:ir_unit+1]
        
        if il_unit > ir_unit:
            print(f"Error: il_unit of {il_unit} greater than ir_unit of {ir_unit}")
            return None
        
        # Move Right
        if direction in ['BOTH', 'LEFT']:
            while il_unit < ir_unit:
                # We must check if the current token is not allowed. If it's
                # not allowed, we contract (remove).
                token = doc[il_unit]

                # include = True means that we want the tokens that match
                # the speech and/or literals in the contracted unit.
                
                # include = False means that we don't want the tokens that
                # match the speech and/or literals in the contracted unit.
                
                # Case 1: include = True, in_set = True
                # We have a token that's meant to be included in the set.
                # However, we're contracting, which means we would end up
                # removing the token if we continue. Therefore, we break.
                
                # Case 2: include = False, in_set = False
                # We have a token that's not in the set which defines the
                # tokens that aren't meant to be included. Therefore, we 
                # have a token that is meant to be included. If we continue,
                # we would end up removing this token. Therefore, we break.
                
                # Default:
                # If we have a token that's in the set (in_set=True) of
                # tokens we're not supposed to include in the contracted 
                # unit (include=False), we need to remove it. Likewise, if
                # we have a token that's not in the set (in_set=False) of
                # tokens to include in the contracted unit (include=True),
                # we need to remove it.
                
                in_set = token.pos_ in speech or token.lower_ in literals
                if include == in_set:
                    break

                # The token is valid, thus we continue.
                il_unit += 1

        # Move Left      
        if direction in ['BOTH', 'RIGHT']:
            while ir_unit > il_unit:
                token = doc[ir_unit]

                # The token is invalid and we
                # stop contracting.
                in_set = token.pos_ in speech or token.lower_ in literals
                if include == in_set:
                    break

                # The token is valid and we continue.
                ir_unit -= 1

        assert il_unit <= ir_unit
        contracted_unit = doc[il_unit:ir_unit+1]

        if verbose and VERBOSE_LEVEL >= 1:
            print(f"Contracted Unit of '{UNIT}': {contracted_unit}")
        
        return contracted_unit

In [27]:
def break_text(text: str, return_type: Literal["Flat", "TextFlat", "Text", "TextAdd"]) -> Union[List[List[Tuple[int, int]]], List[Tuple[int, int]], List[List[str]], List[str]]:
    enclosures = {
        "(":")", 
        "[":"]",
        "{":"}"
    }
    
    # This contains the text that's not inside
    # any enclosure.
    base = []

    # This contains the text that's inside
    # an enclosure.
    groups = []
    
    # This is used for building groups, which can
    # have a nested structure.
    stacks = []
    
    # These are the pairs of characters that
    # define the enclosure (parenthetical).
    openers = list(enclosures.keys())
    closers = list(enclosures.values())
    
    # This contains the opening characters of the groups 
    # that are currently open (e.g. '(', '['). We use it 
    # so that we know whether to open or close a group.
    opened = []
    
    for i, char in enumerate(text):
        # Open Group
        if char in openers:
            stacks.append([])
            opened.append(char)
        # Close Group
        elif opened and char == enclosures.get(opened[-1], ""):
            groups.append(stacks.pop())
            opened.pop()
        # Add to Group
        elif opened:
            stacks[-1].append(i)
        # Add to Base Text
        else:
            base.append(i)
    
    # We close the remaining groups that have not
    # been closed.
    while stacks:
        groups.append(stacks.pop())
        
    # Cluster Groups' Indices
    # A list in the lists of indices (where each list represents a group of text) could have 
    # an interruption (e.g. [0, 1, 2, 10, 15]) because of a parenthetical. So, we cluster the
    # indices in each list to make the output more useful (e.g. [(0, 3), (10, 16)]). As you
    # can see, we've adjusted some indices for ease-of-use.
    lists_of_indices = [*groups, base]        
    lists_of_clustered_indices = []

    for list_of_indices in lists_of_indices:
        if not list_of_indices:
            continue

        # We start off with a single cluster that is made up of the
        # first index. If the next index follows the first index, 
        # we continue the cluster. If it doesn't, we create a new cluster.
        clustered_indices = [[list_of_indices[0], list_of_indices[0] + 1]]
        
        for index in list_of_indices[1:]:
            if clustered_indices[-1][1] == index:
                clustered_indices[-1][1] = index + 1
            else:
                clustered_indices.append([index, index + 1])

        # Add Clustered Indices
        lists_of_clustered_indices.append(clustered_indices)

    if return_type in ["Flat", "TextFlat"]:
        flat_clusters = []
        # We are placing each cluster of indices into one list.
        # This removes the context of the larger parenthetical,
        # but the context may be cumbersome instead of useful.
        for list_of_clustered_indices in lists_of_clustered_indices:
            for clustered_indices in list_of_clustered_indices:
                flat_clusters.append(clustered_indices)
        lists_of_clustered_indices = flat_clusters
    
        if return_type == "TextFlat":
            return [text[cluster[0]:cluster[1]] for cluster in lists_of_clustered_indices]
    
    if return_type in ["Text", "TextAdd"]:
        lists_of_clustered_text = [[text[cluster[0]:cluster[1]] for cluster in clusters] for clusters in lists_of_clustered_indices]
        if return_type == "TextAdd":
            return ["".join(list_of_clustered_text) for list_of_clustered_text in lists_of_clustered_text]
        return lists_of_clustered_text

    return lists_of_clustered_indices

In [28]:
def clean_text(text: str) -> str:
    # 1. Delete Inside and Outside Space
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([?.!,])", r"\1", text)
    text = text.strip()

    # 2. Delete Outside Non-Letters
    while text:
        start_len = len(text)
        # Remove Leading Non-Alphanumeric Character
        if text and not text[0].isalnum():
            text = text[1:]
        # Remove Trailing Non-Alphanumeric Character
        if text and not text[-1].isalnum():
            text = text[:-1]
        # No Changes Made
        if start_len == len(text):
            break
    
    return text

In [29]:
CACHE_PLURALIZE_TEXT = {}

def pluralize_text(text: str) -> str:
    if text in CACHE_PLURALIZE_TEXT:
        return CACHE_PLURALIZE_TEXT[text]
    
    split_text = text.split()
    plural_end = inflector.plural_noun(split_text[-1])

    if not plural_end:
        CACHE_PLURALIZE_TEXT[text] = False
        return False

    split_text[-1] = plural_end
    plural_text = " ".join(split_text)

    CACHE_PLURALIZE_TEXT[text] = plural_text
    return plural_text

In [30]:
CACHE_SINGULARIZE_TEXT = {}

def singularize_text(text: str) -> str:
    if text in CACHE_SINGULARIZE_TEXT:
        return CACHE_SINGULARIZE_TEXT[text]
    
    split_text = text.split()
    singular_end = inflector.singular_noun(split_text[-1])
    
    if not singular_end:
        CACHE_SINGULARIZE_TEXT[text] = False
        return False

    split_text[-1] = singular_end
    singular_text = " ".join(split_text)

    CACHE_SINGULARIZE_TEXT[text] = singular_text
    return singular_text

In [31]:
def tn_search_strings(ents: List[Span]) -> List[str]:
    search_strings = []
    
    for ent in ents:
        split_text = break_text(ent.text.lower(), return_type="TextAdd")
        split_text = [clean_text(text) for text in split_text]
        search_strings.extend(split_text)

        for text in split_text:    
            # Add Base Noun
            base = text.split()[-1]
            search_strings.append(base)

            # Add Singular Version
            s_text = singularize_text(text)
            if s_text:
                search_strings.append(s_text)

            # Add Plural Version
            p_text = pluralize_text(text)
            if p_text:
                search_strings.append(p_text)

    # Remove Duplicates
    search_strings = [*set(search_strings)]
    return search_strings

In [32]:
def find_tn_search_strings(doc: Doc, search_strings: List[str]) -> List[Span]:
    spans = []
    text = doc.text.lower()
    
    for string in search_strings:
        matches = re.finditer(re.escape(string), text, re.IGNORECASE)
        
        for l_char_index, r_char_index, matched_text in [(match.start(), match.end(), match.group()) for match in matches]:
            # The full word must match, not just a substring inside of it.
            # So, if the species we're looking for is "ant", only "ant"
            # will match -- not "pants" or "antebellum". Therefore, the
            # characters to the left and right of the matched string cannot
            # be letters.
            l_char_is_letter = l_char_index > 0 and text[l_char_index-1].isalpha()
            r_char_is_letter = r_char_index < len(text) and text[r_char_index].isalpha()
            
            if l_char_is_letter or r_char_is_letter or not matched_text:
                continue
            
            span = doc.char_span(l_char_index, r_char_index, alignment_mode="expand")
            if not span:
                continue
            
            # Expand Species
            # Let's say there's a word like "squirrel". That's a bit ambiguous. 
            # Is it a brown squirrel, a bonobo? If the species is possibly missing
            # information (like an adjective to the left of it), we should expand
            # in order to get a full picture of the species.
            is_short = len(span) == 1 and span[0].pos_ == "NOUN"
            is_scientific = re_is_scientific_name_abbrev(span.text) or found_scientific_name(span.text) or found_genus(span.text)

            # Remove Outer Symbols
            # There are times where a species is identified with a parenthesis
            # nearby. Here, we remove that parenthesis (and any other symbols).
            span = contract_unit(
                doc,
                il_unit=span.start, 
                ir_unit=span.end-1, 
                speech=["PUNCT", "SYM", "DET", "PART"],
                include=False,
                verbose=False
            )
            
            if is_short and not is_scientific:
                span = expand_unit(
                    doc,
                    il_unit=span.start, 
                    ir_unit=span.end-1,
                    il_boundary=0,
                    ir_boundary=len(doc),
                    speech=["ADJ", "PROPN", "NOUN"],
                    literals=["-"],
                    include=True,
                    direction="LEFT",
                    verbose=False
                )

            # Remove Outer Symbols (Again)
            # There are times where a species is identified with a parenthesis
            # nearby. Here, we remove that parenthesis (and any other symbols).
            span = contract_unit(
                doc,
                il_unit=span.start, 
                ir_unit=span.end-1, 
                speech=["PUNCT", "SYM", "DET", "PART"],
                include=False,
                verbose=False
            )
            
            # A species must have a noun or a
            # proper noun. This may help discard
            # bad results.
            letter_found = False
            for token in span:
                if token.pos_ in ["NOUN", "PROPN"]:
                    letter_found = True
                    break

            if not letter_found:
                continue

            # Adding Species
            spans.append(span)

    spans = filter_spans(spans)
    return spans

In [33]:
def find_ents_tn(doc: Doc) -> List[Span]:
    search_strings = tn_search_strings(doc.ents)
    ents = find_tn_search_strings(doc, search_strings)
    return ents

In [34]:
def find_ents_db(doc: Doc) -> List[Span]:
    ents = []
        
    cache = set()
    scientific_tokens = []
    nouns = ["NOUN", "PROPN", "X"]
    
    # Look for Scientific Names
    # These are in pairs of two, which makes the search easier.
    # The pair of two must contain a noun and follow the capitalization
    # format of scientific names.
    i = 0
    while i < len(doc) - 1:
        # Check for Genus
        token = doc[i:i+1]
        if token[0].pos_ in nouns and token.text not in cache and found_genus(token.text):
            cache.add(token.text)
            ents.append(token)
            scientific_tokens.append(token[0])

        # Check for Scientific Name
        j = i + 1
        if doc[i].pos_ not in nouns and doc[j].pos_ not in nouns:
            i += 1
            continue
        
        span = doc[i:j+1]
        text = span.text
    
        if text in cache:
            ents.append(span)
            scientific_tokens.extend(list(span))
            i += 2
        elif found_scientific_name(text):
            cache.add(text)
            cache.add(abbreviate_scientific_name(text))
            ents.append(span)
            scientific_tokens.extend(list(span))
            i += 2
        elif re_is_scientific_name_abbrev(text):
            cache.add(text)
            ents.append(span)
            scientific_tokens.extend(list(span))
            i += 2
        else:
            i += 1


    # Look for Vernacular Names
    i = len(doc) - 1
    while i >= 0:
        if doc[i].pos_ != "NOUN" or doc[i] in scientific_tokens:
            i -= 1
            continue

        span = doc[i:i+1]
        text = span.text
        
        if text in cache or found_vernacular_name(text):
            cache.add(text)

            # 1. Contract
            span = contract_unit(
                doc,
                il_unit=span.start, 
                ir_unit=span.end-1, 
                speech=["PUNCT", "SYM", "DET", "PART"],
                include=False,
                verbose=False
            )

            # 2. Expand
            span = expand_unit(
                doc,
                il_unit=span.start, 
                ir_unit=span.end-1,
                il_boundary=0,
                ir_boundary=len(doc),
                speech=["ADJ", "PROPN", "NOUN"],
                literals=["-"],
                include=True,
                direction="LEFT",
                verbose=False
            )

            # 3. Contract
            span = contract_unit(
                doc,
                il_unit=span.start, 
                ir_unit=span.end-1, 
                speech=["PUNCT", "SYM", "DET", "PART"],
                include=False,
                verbose=False
            )

            ents.append(span)
        
        i -= 1
    
    ents = filter_spans(ents)
    return ents

In [35]:
def find_ents(doc: Doc) -> List[Span]:
    ents_tax = find_ents_tn(doc)
    ents_col = find_ents_db(doc)

    ents = [
        *ents_tax, 
        *ents_col
    ]
    ents = filter_spans(ents)

    # Clean-Up
    i = 0
    while i < len(ents):
        # These do not count!
        if "species" in ents[i].text.lower():
            ents.pop(i)
            continue
        i += 1
    
    return ents

In [36]:
def a_abbreviates_b(span_a: Span, span_b: Span) -> bool:
    if len(span_a) < 2 or len(span_b) < 2:
        return False
    
    text_a = f"{span_a[-2]} {span_a[-1]}"
    text_b = f"{span_b[-2]} {span_b[-1]}"

    if text_a == text_b:
        return True

    return text_a == abbreviate_scientific_name(text_b)

In [37]:
CACHE_B_SUBSTITUTES_A = {}

def b_substitutes_a(span_a: Span, span_b: Span, substitutions: Dict[str, List[str]]) -> bool:
    key = (span_a.text.lower(), span_b.text.lower())
    key_rev = (span_b.text.lower(), span_a.text.lower())
    
    if key in CACHE_B_SUBSTITUTES_A:
        return CACHE_B_SUBSTITUTES_A[key]
    
    singular_tags = ["NN", "NNP"]
    
    # Span A's Text Versions
    text_a = span_a.text.lower()
    plural_text_a = span_a[-1].tag_ in singular_tags and pluralize_text(text_a)
    singular_text_a = span_a[-1].tag_ not in singular_tags and singularize_text(text_a)

    # Span B's Text Versions
    text_b = span_b.text.lower()
    plural_text_b = span_b[-1].tag_ in singular_tags and pluralize_text(text_b)
    singular_text_b = span_b[-1].tag_ not in singular_tags and singularize_text(text_b)
    
    text_versions_a = set([text for text in [text_a, plural_text_a, singular_text_a] if text])
    text_versions_b = set([text for text in [text_b, plural_text_b, singular_text_b] if text])
    
    for text in text_versions_a:
        if text and text_versions_b.intersection(substitutions.get(text, [])):
            CACHE_B_SUBSTITUTES_A[key] = True
            CACHE_B_SUBSTITUTES_A[key_rev] = True
            return True

    # Check DB Substitutions
    if found_scientific_name(text_a):
        subs = find_scientific_name_substitutions(text_a)
        if text_versions_b.intersection(subs):
            CACHE_B_SUBSTITUTES_A[key] = True
            CACHE_B_SUBSTITUTES_A[key_rev] = True
            return True
    else:
        for text in text_versions_a:
            subs = find_vernacular_name_substitutions(text)
            if text_versions_b.intersection(subs):
                CACHE_B_SUBSTITUTES_A[key] = True
                CACHE_B_SUBSTITUTES_A[key_rev] = True
                return True
    
    CACHE_B_SUBSTITUTES_A[key] = False
    CACHE_B_SUBSTITUTES_A[key_rev] = False
    return False

In [67]:
CACHE_A_MODIFIES_B = {}

def a_modifies_b(span_a: Span, span_b: Span) -> bool:
    key = (span_a.text.lower(), span_b.text.lower())
    key_rev = (span_b.text.lower(), span_a.text.lower())
    
    if key in CACHE_A_MODIFIES_B:
        return CACHE_A_MODIFIES_B[key]

    if re.search(rf"\b{span_b.text.lower()}\b", span_a.text.lower()):
        CACHE_A_MODIFIES_B[key] = True
        return True
    
    # Base (Ending) Nouns in A
    base_nouns_a = []
    for token in reversed(span_a):
        if token.pos_ not in ["NOUN", "PROPN"]:
            break
        base_nouns_a.insert(0, token.text.lower())

    # Base (Ending) Nouns in B
    base_nouns_b = []
    for token in reversed(span_b):
        if token.pos_ not in ["NOUN", "PROPN"]:
            break
        base_nouns_b.insert(0, token.text.lower())

    if not base_nouns_a or not base_nouns_b:
        CACHE_A_MODIFIES_B[key] = False
        return False

    base_a = " ".join(base_nouns_a)
    base_b = " ".join(base_nouns_b)

    singular_tags = ["NN", "NNP"]
    
    base_versions_a = [version for version in [base_a, span_a[-1].tag_ in singular_tags and pluralize_text(base_a), span_a[-1].tag_ not in singular_tags and singularize_text(base_a)] if version]
    base_versions_b = [version for version in [base_b, span_b[-1].tag_ in singular_tags and pluralize_text(base_b), span_b[-1].tag_ not in singular_tags and singularize_text(base_b)] if version]
    
    for text_a in base_versions_a:
        for text_b in base_versions_b:
            if text_a in text_b:
                CACHE_A_MODIFIES_B[key_rev] = True
                
            if text_b in text_a:
                CACHE_A_MODIFIES_B[key] = True
                return True

    CACHE_A_MODIFIES_B[key] = False
    return False

In [39]:
def a_inflects_b(span_a: Span, span_b: Span) -> bool:
    text_a = span_a.text.lower()
    text_b = span_b.text.lower()

    singular_tags = ["NN", "NNP"]
    if span_a[-1].tag_ not in singular_tags and span_b[-1].tag_ in singular_tags:
        singular_text_a = singularize_text(text_a)
        return singular_text_a == text_b
    
    return False

In [40]:
def find_substitutions(spans: List[Span], doc: Doc) -> Dict[str, List[str]]:
    spans = [*spans]
    spans.sort(key=lambda span: span.start)

    # The text provides information about 
    # names that will be used interchangeably,
    # like "predatory crab (Carcinus maenas)".
    substitutions = {}

    for i, span in enumerate(spans):
        if i + 1 >= len(spans):
            break

        next_span = spans[i+1]
        
        is_substitute = False
        
        if next_span.start - span.end == 1 and next_span.end < len(doc):
            bef_next_span = doc[next_span.start-1].text.lower()
            aft_next_span = doc[next_span.end].text.lower()
            is_substitute = bef_next_span == "(" and aft_next_span in [".", ")"]
        else:
            is_substitute = next_span.start - span.end == 0

        if not is_substitute:
            continue
        
        span_text = span.text.lower()
        next_span_text = next_span.text.lower()
        
        if span_text not in substitutions:
            substitutions[span_text] = []
        
        if next_span_text not in substitutions:
            substitutions[next_span_text] = []
        
        substitutions[span_text].append(next_span_text)
        substitutions[next_span_text].append(span_text)
    
    return substitutions

In [41]:
def find_substitutions_2(spans: List[Span], doc: Doc) -> Dict[str, List[str]]:
    spans = [*spans]
    spans.sort(key=lambda span: span.start)
    
    token_to_span = {}
    for span in spans:
        for token in span:
            token_to_span[token] = span
    
    # The text provides information about 
    # names that will be used interchangeably,
    # like "predatory crab (Carcinus maenas)".
    substitutions = {}

    tracking = None
    for token in doc:
        # Neighboring Spans
        if token in token_to_span and token_to_span[token][-1] == token:
            token_span = token_to_span[token]
            token_next = token.i < len(doc) - 1 and token.nbor()
            
            if token_next and token_next in token_to_span:
                token_next_span = token_to_span[token_next]

                token_span_text = token_span.text.lower()
                token_next_span_text = token_next_span.text.lower()
                
                if token_span_text not in substitutions:
                    substitutions[token_span_text] = []

                if token_next_span_text not in substitutions:
                    substitutions[token_next_span_text] = []

                substitutions[token_span_text].append(token_next_span_text)
                substitutions[token_next_span_text].append(token_span_text)

        # Dealing with Parentheses
        # Example 'The Panthera leo (lion) is hungry.'
        if token.text == "(":
            tracking = token
        
        if tracking and token.text == ")":
            # Spans in Parentheses
            tracked_spans = set()
            for i in range(tracking.i + 1, token.i):
                token = doc[i]
                if token in token_to_span:
                    tracked_spans.add(token_to_span[token])

            # Span to Left of Parentheses
            tracking_token = tracking.i != 0 and tracking.nbor(-1)
            if tracking_token and tracking_token in token_to_span:
                tracking_span = token_to_span[tracking_token]

                tracking_span_text = tracking_span.text.lower()
                if tracking_span_text not in substitutions:
                    substitutions[tracking_span_text] = []
        
                for span in tracked_spans:
                    span_text = span.text.lower()
                    if span_text not in substitutions:
                        substitutions[span_text] = []
                    
                    substitutions[span_text].append(tracking_span_text)
                    substitutions[tracking_span_text].append(span_text)
            
            tracking = None
    
    return substitutions

In [42]:
def sort_ents(ents: List[Span], doc: Doc) -> List[List[Span]]:
    ents = [*ents]
    ents.sort(key=lambda ent: ent.text.lower())
    sorted_ents = [[ent] for ent in ents]
    
    # 1. Group by Text
    i = 0
    while i < len(sorted_ents) - 1:
        if sorted_ents[i][0].text.lower() == sorted_ents[i+1][0].text.lower():
            sorted_ents[i].extend(sorted_ents.pop(i+1))
        else:
            i += 1
    
    # 2. Group by Name
    # We check scientific names, accounting for 
    # abbreviations. We also check vernacular names, 
    # accounting for singular and plural differences.
    i = 0
    while i < len(sorted_ents) - 1:
        j = i + 1
        
        while j < len(sorted_ents):
            conditions = [
                a_abbreviates_b(sorted_ents[i][0], sorted_ents[j][0]),
                a_abbreviates_b(sorted_ents[j][0], sorted_ents[i][0]),
                a_inflects_b(sorted_ents[i][0], sorted_ents[j][0]),
                a_inflects_b(sorted_ents[j][0], sorted_ents[i][0])
            ]
            
            if any(conditions):
                sorted_ents[i].extend(sorted_ents.pop(j))
                continue
            
            j += 1
        
        i += 1
    
    # 3. Group by Substitution
    # As a Panthera leo can also be called a lion,
    # these two entities should be placed together.
    substitutions = find_substitutions(ents, doc)
    
    i = 0
    num_iterations = 0
    while i < len(sorted_ents):
        matched = []
        
        # We compare the group to all the
        # other groups.
        j = 0
        while j < len(sorted_ents):
            if j == i:
                j += 1
                continue

            for ent_i, ent_j in [(ent_i, ent_j) for ent_i in sorted_ents[i] for ent_j in sorted_ents[j]]:
                num_iterations += 1
                if (
                    re_is_scientific_name_abbrev(ent_i.text) or 
                    re_is_scientific_name_abbrev(ent_j.text)
                ):
                    continue

                conditions = [
                    b_substitutes_a(ent_i, ent_j, substitutions),
                    b_substitutes_a(ent_j, ent_i, substitutions),
                    a_modifies_b(ent_i, ent_j),
                    a_modifies_b(ent_j, ent_i)
                ]

                if any(conditions):
                    matched.append(j)
                    break
            
            j += 1

        # The given group matches with
        # only one other group.
        if len(matched) == 1:
            sorted_ents[i].extend(sorted_ents.pop(matched[0]))
        # If the given group matches multiple
        # groups, we remove it.
        elif len(matched) > 1:
            sorted_ents.pop(i)
        # There are no matches.
        # So, we continue to the next group.
        else:
            i += 1
    
    return sorted_ents

In [43]:
# Types of a Species' Name
# This is used in the sorting function to
# limit the number of function calls.
GENUS = "GENUS"
VERNACULAR = "VERNACULAR"
SCIENTIFIC = "SCIENTIFIC"
SCIENTIFIC_ABBREV = "SCIENTIFIC_ABBREVIATED"

In [44]:
# Deduplicate Tuples of Spans
# Similarly, this is used in the sorting
# function to limit the number of calls.
# Think of the tuples as parameters to
# a function; duplicate tuples indicates
# an unnecessary function call. Hence, its
# removal.
def deduplicate_tuples(tuples: Tuple[Span, Span]) -> Tuple[Span, Span]:
    log = {}
    for spans in tuples:
        log[(spans[0].text, spans[1].text)] = spans
    return [*log.values()]    

In [76]:
def sort_ents_2(ents: List[Span], doc: Doc) -> List[List[Span]]:
    ents = [*ents]
    ents.sort(key=lambda ent: ent.text.lower())

    print("Sorted Entities", ents, "\n\n")
    
    sorted_ents = [[ent] for ent in ents]
    
    # 1. Group by Text
    # O(n)
    i = 0
    while i < len(sorted_ents) - 1:
        if sorted_ents[i][0].text.lower() == sorted_ents[i+1][0].text.lower():
            sorted_ents[i].extend(sorted_ents.pop(i+1))
        else:
            i += 1

    print("Entities Grouped by Text", sorted_ents, "\n\n")
    
    # Fix Groups w/ Type
    # O(n)
    i = 0
    while i < len(sorted_ents):
        ent_type = VERNACULAR
        if re_is_scientific_name_abbrev(sorted_ents[i][0].text):
            ent_type = SCIENTIFIC_ABBREV
        elif re_is_scientific_name(sorted_ents[i][0].text) and found_scientific_name(sorted_ents[i][0].text):
            ent_type = SCIENTIFIC
        elif re_is_genus(sorted_ents[i][0].text) and found_genus(sorted_ents[i][0].text):
            ent_type = GENUS
        elif i > 0:
            ok = [*sorted_ents[i-1][VERNACULAR], *sorted_ents[i-1][SCIENTIFIC], *sorted_ents[i-1][SCIENTIFIC_ABBREV], *sorted_ents[i-1][GENUS]][0].text.lower()
            print(ok)
            search = rf"{sorted_ents[i][0].text.lower()}"
            print(search)
            re.search(search, rf"\b{ok}\b")
            ent_type = sorted_ents[i-1]["type"]
        else:
            
        
        tagged_ent = {
            SCIENTIFIC: [],
            SCIENTIFIC_ABBREV: [],
            VERNACULAR: [],
            GENUS: []
        }
        tagged_ent[ent_type] = [*sorted_ents[i]]
        tagged_ent["type"] = ent_type
        sorted_ents[i] = tagged_ent
        
        i += 1

    # print("Entities with Type", [[_[SCIENTIFIC] + _[SCIENTIFIC_ABBREV] + _[GENUS] + _[VERNACULAR]] for _ in sorted_ents], "\n\n")
    print("Typed Groups")
    for group in sorted_ents:
        print(group)
    
    # 2. Group by Name
    # We check scientific names, accounting for 
    # abbreviations. We also check vernacular names, 
    # accounting for singular and plural differences.
    i = 0
    while i < len(sorted_ents) - 1:
        matched = []
        j = 0
        while j < len(sorted_ents):
            if j == i:
                j += 1
                continue
            
            merge = False

            if sorted_ents[i][SCIENTIFIC] and sorted_ents[j][SCIENTIFIC_ABBREV]:
                merge = a_abbreviates_b(
                    sorted_ents[j][SCIENTIFIC_ABBREV][0],
                    sorted_ents[i][SCIENTIFIC][0]
                )
            
            if not merge and sorted_ents[i][VERNACULAR] and sorted_ents[j][VERNACULAR]:
                merge = a_inflects_b(
                    sorted_ents[i][VERNACULAR][0], 
                    sorted_ents[j][VERNACULAR][0]
                )
            
            if not merge and sorted_ents[i][SCIENTIFIC] and sorted_ents[j][GENUS]:
                merge = sorted_ents[j][GENUS][0].text.lower() in sorted_ents[i][SCIENTIFIC][0].text.lower()
            
            if merge:
                matched.append(j)
            
            j += 1

        if len(matched) == 1:
            j = matched[0]
            sorted_ents[i] = {
                SCIENTIFIC: sorted_ents[i][SCIENTIFIC] + sorted_ents[j][SCIENTIFIC],
                SCIENTIFIC_ABBREV: sorted_ents[i][SCIENTIFIC_ABBREV] + sorted_ents[j][SCIENTIFIC_ABBREV],
                VERNACULAR: sorted_ents[i][VERNACULAR] + sorted_ents[j][VERNACULAR],
                GENUS: sorted_ents[i][GENUS] + sorted_ents[j][GENUS]
            }
            sorted_ents.pop(j)
        
        elif len(matched) > 1:
            sorted_ents.pop(i)
        else:
            i += 1
        
    print("Entities Grouped by Name", [[_[SCIENTIFIC] + _[SCIENTIFIC_ABBREV] + _[GENUS] + _[VERNACULAR]] for _ in sorted_ents], "\n\n")
    
    # 3. Group by Modification
    # So, pairs like "tadpoles" and "Hyla tadpoles"
    # or "dog" and "red dog" can be grouped.
    # O(n^2) BAD
    substitutions = find_substitutions_2(ents, doc)
    
    i = 0
    while i < len(sorted_ents):
        matched = []
        # We compare the group to all the
        # other groups.
        j = 0
        while j < len(sorted_ents):
            if j == i:
                j += 1
                continue

            merge = False

            # Check Modifications
            for ent_type in [VERNACULAR, SCIENTIFIC, SCIENTIFIC_ABBREV, GENUS]:
                if not sorted_ents[i][ent_type] and sorted_ents[j][ent_type]:
                    continue
                
                combos = [(ent_i, ent_j) for ent_j in sorted_ents[j][ent_type] for ent_i in sorted_ents[i][ent_type]]
                combos = deduplicate_tuples(combos)
                
                for combo in combos:
                    if a_modifies_b(combo[0], combo[1]):
                        merge = True
                        break

                if merge:
                    break
            
            if merge:
                matched.append(j)
            
            j += 1

        # Matched w/ 1 Group
        if len(matched) == 1:
            j = matched[0]
            sorted_ents[i] = {
                SCIENTIFIC: sorted_ents[i][SCIENTIFIC] + sorted_ents[j][SCIENTIFIC],
                SCIENTIFIC_ABBREV: sorted_ents[i][SCIENTIFIC_ABBREV] + sorted_ents[j][SCIENTIFIC_ABBREV],
                VERNACULAR: sorted_ents[i][VERNACULAR] + sorted_ents[j][VERNACULAR],
                GENUS: sorted_ents[i][GENUS] + sorted_ents[j][GENUS]
            }
            sorted_ents.pop(j)

        # Matched w/ Multiple
        # We remove the group.
        elif len(matched) > 1:
            sorted_ents.pop(i)

        # No Matches
        else:
            i += 1

    # 4. Group by Substitution
    # As a Panthera leo can also be called a lion,
    # these two entities should be placed together.
    i = 0
    while i < len(sorted_ents):
        matched = []
        # We compare the group to all the
        # other groups.
        j = 0
        while j < len(sorted_ents):
            if j == i:
                j += 1
                continue

            merge = False
            
            # Check Substitutions
            # If we're already merging, there's
            # no point in doing this.
            ents_i = [*sorted_ents[i][VERNACULAR], *sorted_ents[i][SCIENTIFIC], *sorted_ents[i][SCIENTIFIC_ABBREV], *sorted_ents[i][GENUS]]
            ents_j = [*sorted_ents[j][VERNACULAR], *sorted_ents[j][SCIENTIFIC], *sorted_ents[j][SCIENTIFIC_ABBREV], *sorted_ents[j][GENUS]]
            
            combos = [(ent_i, ent_j) for ent_j in ents_j for ent_i in ents_i]
            combos = deduplicate_tuples(combos)
            
            for combo in combos:
                if b_substitutes_a(combo[0], combo[1], substitutions):
                    merge = True
                    break
            
            if merge:
                matched.append(j)
            
            j += 1

        # Matched w/ 1 Group
        if len(matched) == 1:
            j = matched[0]
            sorted_ents[i] = {
                SCIENTIFIC: sorted_ents[i][SCIENTIFIC] + sorted_ents[j][SCIENTIFIC],
                SCIENTIFIC_ABBREV: sorted_ents[i][SCIENTIFIC_ABBREV] + sorted_ents[j][SCIENTIFIC_ABBREV],
                VERNACULAR: sorted_ents[i][VERNACULAR] + sorted_ents[j][VERNACULAR],
                GENUS: sorted_ents[i][GENUS] + sorted_ents[j][GENUS]
            }
            sorted_ents.pop(j)

        # Matched w/ Multiple
        # We remove the group.
        elif len(matched) > 1:
            sorted_ents.pop(i)

        # No Matches
        else:
            i += 1
    
    return sorted_ents

In [46]:
def clear_cache():
    global CACHE_SEARCH_SCIENTIFIC_NAME
    global CACHE_SEARCH_VERNACULAR_NAME
    global CACHE_SEARCH_GENUS
    global CACHE_FIND_SCIENTIFIC_NAME_SUBSTITUTIONS
    global CACHE_FIND_VERNACULAR_NAME_SUBSTITUTIONS
    global CACHE_FIND_GENUS_SUBSTITUTIONS
    global CACHE_B_SUBSTITUTES_A
    global CACHE_A_MODIFIES_B
    global CACHE_SINGULARIZE_TEXT
    global CACHE_PLURALIZE_TEXT
    
    CACHE_SEARCH_SCIENTIFIC_NAME = {}
    CACHE_SEARCH_VERNACULAR_NAME = {}
    CACHE_SEARCH_GENUS = {}
    CACHE_FIND_SCIENTIFIC_NAME_SUBSTITUTIONS = {}
    CACHE_FIND_VERNACULAR_NAME_SUBSTITUTIONS = {}
    CACHE_FIND_GENUS_SUBSTITUTIONS = {}
    CACHE_B_SUBSTITUTES_A = {}
    CACHE_A_MODIFIES_B = {}
    CACHE_SINGULARIZE_TEXT = {}
    CACHE_PLURALIZE_TEXT = {}

In [47]:
def passes(doc: Doc) -> bool:
    ents = find_ents(doc)
    sorted_ents = sort_ents_2(ents, doc)
    return (len(sorted_ents) >= 3, sorted_ents)

In [48]:
nlp = TaxoNERD(prefer_gpu=False).load(model="en_ner_eco_biobert", exclude=["tok2vec", "parser", "lemmatizer"])

In [49]:
df = pd.read_csv("ScreenedPapers-3.csv")
print(f"Shape: {df.shape}")

Shape: (27783, 55)


In [77]:
# txt = df.Abstract[16]
# doc = nlp(txt)
ents = find_ents(doc)
ent_groups = sort_ents_2(ents, doc)

Sorted Entities [A. maculatum, A. maculatum, A. maculatum, A. maculatum, A. maculatum larvae, A. talpodeum, A. talpoideum, A. talpoideum, A. talpoideum larvae, Ambystoma talpoideum, fish, fish, fish, fish, L. macrochirus, L. macrochirus, L. macrochirus, L. macrochirus, larvae, larvae, larvae, larvae, larvae, Larvae, larvae, Lepomis macrochirus, salamander larvae, salamander larvae] 


Entities Grouped by Text [[A. maculatum, A. maculatum, A. maculatum, A. maculatum], [A. maculatum larvae], [A. talpodeum], [A. talpoideum, A. talpoideum], [A. talpoideum larvae], [Ambystoma talpoideum], [fish, fish, fish, fish], [L. macrochirus, L. macrochirus, L. macrochirus, L. macrochirus], [larvae, larvae, larvae, larvae, larvae, Larvae, larvae], [Lepomis macrochirus], [salamander larvae, salamander larvae]] 


a. maculatum
a. maculatum larvae
a. talpoideum
a. talpoideum larvae
ambystoma talpoideum
fish
l. macrochirus
larvae
lepomis macrochirus
salamander larvae
Typed Groups
{'SCIENTIFIC': [], 'SCIENT

In [ ]:
mask = []
errors = []

with open("ScreenByEntitiesMask.pickle", 'rb') as file:
    mask = pickle.load(file)

counts = {None: 0, True: 0, False: 0}
for flag in mask:
    counts[flag] += 1

texts = df.Abstract.tolist()[len(mask):]
number_texts = len(df.Abstract.tolist())
outputs = {}

t0 = time.time()
i = len(mask)

for doc in nlp.pipe(texts, batch_size=64):
    # Auto-Save
    if (len(mask) + 1) % 20:
        with open('ScreenByEntitiesMask-2.pickle', 'wb') as file:
            pickle.dump(mask, file)
    
    flag = None
    
    try:
        passed, ents = passes(doc)
        flag = passed
        outputs[i] = ents
        
        clear_cache()
    except Exception as e:
        flag = None
        errors.append((i, e))

    counts[flag] += 1
    mask.append(flag)
    
    t1 = time.time()
    elapsed = int(t1 - t0)
    m, s = divmod(elapsed, 60)
    h, m = divmod(m, 60)
    
    clear_output(wait=True)
    print(f"{i+1}/{number_texts} ({h:d}:{m:02d}:{s:02d})")
    print(f"Number Included: {counts[True]}")
    print(f"Number Excluded: {counts[False]}")
    print(f"Number Errors: {counts[None]}")

    i += 1

In [ ]:
with open('ScreenByEntitiesOutputs-2.pickle', 'wb') as file:
    mod_outputs = {}
    for k, v in outputs.items():
        mod_outputs[k] = []
        for ents in v:
            mod_outputs[k].append([
                *[ent.text for ent in ents[SCIENTIFIC]],
                *[ent.text for ent in ents[SCIENTIFIC_ABBREV]],
                *[ent.text for ent in ents[VERNACULAR]],
                *[ent.text for ent in ents[GENUS]]
            ])
    pickle.dump(mod_outputs, file)